## Reopening the index

In a new Python session, you can reopen the index without re-computing
embeddings:


In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Now we can search:

In [3]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [6]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [7]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [8]:
results

[{'id': '04440cab11',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': 'Kestra AI Copilot replies "I can only assist with creating Kestra flows" — how do I fix it?',
  'answer': 'This message means the AI Copilot didn\'t get a valid Gemini API key, so it falls back to a canned refusal. In Kestra\'s Open Source edition the Copilot only supports Gemini, and it reads the **plain** `GEMINI_API_KEY` variable (not the base64-encoded `SECRET_GEMINI_API_KEY` that the flows use).\n\nMake sure you exported the plain key before starting Kestra, then restart it:\n\n```bash\nexport GEMINI_API_KEY="your-gemini-api-key-here"\ndocker compose up -d\n```\n\nIf it still fails, the key is usually missing, mistyped, or rate-limited:\n\n- Confirm the variable is actually set in the shell you ran `docker compose up` from (`echo $GEMINI_API_KEY`).\n- Generate a fresh key in [Google AI Studio](https://aistudio.google.com/app/apikey).\n- If you\'ve been running the agent/multi-a

We still load the embedding model to encode the query, but we don't
re-embed all the documents. No `fit` call needed, because the index is
already built and waiting on disk.

This is the same two-process split we used for text search in module 1.
One process ingests and builds the index, another queries it.

It matters more here than with text search. Embedding the whole dataset
takes about a minute.\
We don't want a user waiting that long when the
app starts up. We pay that cost once during ingestion, and the query
side starts up instantly.

## Using sqlitesearch vector search in RAG

We'll use the `RAGVector` class we defined in the
[previous lesson](06-rag-vector.md). It overrides the `search` method
to embed the query and use vector search.

Set up the OpenAI client and create the assistant:

In [9]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [11]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still sign up and start learning. If you want a certificate, make sure you submit your project while submissions are still open.'

When you're done, close the connection:

In [12]:
vs_index.close()

## Comparing minsearch and sqlitesearch for vector search

Here is how the two compare:

- minsearch `VectorSearch`: in-memory (numpy), exact cosine similarity,
  must re-compute embeddings on startup, good for experiments and
  notebooks
- sqlitesearch `VectorSearchIndex`: persistent (SQLite `.db` file), ANN
  (LSH/IVF/HNSW) with exact rerank, can open an existing index, good
  for projects and persistence

This is probably the last you'll hear of sqlitesearch. I built it for
teaching, to show the ingestion-then-deployment split.

It does have a real use though. Its only dependencies are SQLite and
numpy. So it runs on any host that offers a free SQLite database, where
a dedicated vector database would cost extra.\
For most work you'll reach
for something else, which is what we do next.